# Ablation Sampling Exploration

This notebook trains a single model and keeps two perturbation estimands separate:

- **Coverage analysis:** enriches rare `(center marker, perturbed marker, hop)` combinations for heatmaps and full effect distributions.
- **Population analysis:** samples representative center cells to estimate conditional and population-average ablation effects under either equal-cell or equal-organoid weighting.

All perturbations use `mode="single"`. For every marker and hop, a separate conditional-effect sample is topped up to a shared minimum where possible. Top-up centers never enter prevalence estimates or another marker's analysis.


**Setup And Settings**


In [17]:
import copy
import json
import random
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root or experiments directory.")
DATA_ROOT = PROJECT_ROOT / "training_data"
sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("torch        =", torch.__version__)
print("cuda         =", torch.cuda.is_available())


PROJECT_ROOT = /home/fmoller/Projects/LearningOrganoids/GraphNN
torch        = 2.8.0+cu128
cuda         = True


In [ ]:
# Data and filtering
DATASET_NAME = "fixed_new"
TARGET_INDICES = [0]
TARGET_INDEX_FOR_ANALYSIS = 0
USE_GLOBAL_FEATURES = True
FILTER_BLACKLISTED_ORGANOIDS = True
SUBTRACT_CONSTANT_GLOBAL_BASELINE = False
TIMEPOINT_FILTER_MODE = "rest"  # "rest", "day3p5", or "all"
DAY3P5_TIMEPOINT = "day3p5"
FILL_MISSING_COMPLEXITY = True
MISSING_COMPLEXITY_GROUP = {
    "dataset": "20251201",
    "timepoint": "day4p5",
    "fill_value": 2.1,
}
SPHERICITY_MAX = 0.92
SPHERICAL_MARKER_DIVERSITY_MIN = 0.5
COMPLEXITY_MIN = 2.0
INTERPOLATE_TARGET_OUTLIERS = True
OUTLIER_CLIP_QUANTILES = (0.005, 0.995)

# Single train/validation split
VAL_FRAC = 0.2
SPLIT_SEED = 42

# Model and training
NUM_LAYERS = 4
HIDDEN_DIM = 4 * 64
DROPOUT = 0.1
NORM = "batch"
RESIDUAL = True
LR = 3e-4
BATCH_SIZE = 128
MAX_EPOCHS = 500
PATIENCE = 30
NUM_WORKERS = 4
EDGE_LOSS_WEIGHT = 0.20
EDGE_LOSS_PARAMS = {
    "weighted": False,
    "alpha": 2.0,
    "normalize_by": "graph_std",
    "clip_weight": 4.0,
}

# Sampling and single-marker ablation
PERTURBATION_MODE = "single"
SUBGRAPH_SEED = 0
PERTURB_BATCH_SIZE = 128
HEATMAP_SAMPLE_SIZE = 2000
HEATMAP_MIN_CENTER_COUNT = 50
HEATMAP_MIN_PAIR_COUNT = 25
HEATMAP_DISPLAY_MIN_CASES = 20
POPULATION_SAMPLE_SIZE = 2000
MIN_MARKER_COUNT_PER_HOP = 100  # Minimum conditional-effect cases for every marker at every hop.
CONVERSION_SPECS = [
    {"key": "lysozyme_to_serotonin", "source": "Lysozyme", "target": "Serotonin", "label": "Lysozyme -> Serotonin"},
    {"key": "serotonin_to_lysozyme", "source": "Serotonin", "target": "Lysozyme", "label": "Serotonin -> Lysozyme"},
]
DISTRIBUTION_CENTER_MARKERS = None  # None plots every center marker.

# Optional robust filtering of per-case prediction changes.
OUTLIER_FILTER_METHOD = "mad"  # None or "mad".
OUTLIER_MAD_Z = 4.0
OUTLIER_MIN_GROUP_SIZE = 20
OUTLIER_TAIL = "both"  # "both", "lower", or "upper".

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = PROJECT_ROOT / "results_experiments" / "ablation_sampling" / RUN_TIMESTAMP
FIGURES_DIR = SAVE_DIR / "figures"
TABLES_DIR = SAVE_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)
torch.manual_seed(SPLIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SPLIT_SEED)


In [19]:
def save_figure(fig, name, *, dpi=250):
    for suffix in ("png", "pdf"):
        fig.savefig(FIGURES_DIR / f"{name}.{suffix}", dpi=dpi, bbox_inches="tight")


def select_target_column(values, target_index=0):
    arr = np.asarray(values)
    if arr.ndim == 1:
        return arr
    return arr[:, int(target_index)]


def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not np.any(valid):
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))


def weighted_quantile(values, quantile, weights=None):
    values = np.asarray(values, dtype=float)
    if weights is None:
        weights = np.ones(len(values), dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[valid]
    weights = weights[valid]
    if len(values) == 0:
        return np.nan
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cumulative = np.cumsum(weights) - 0.5 * weights
    cumulative /= weights.sum()
    return float(np.interp(quantile, cumulative, values))


def robust_outlier_mask(values, weights=None):
    values = np.asarray(values, dtype=float)
    mask = np.zeros(len(values), dtype=bool)
    if OUTLIER_FILTER_METHOD is None:
        return mask
    if OUTLIER_FILTER_METHOD != "mad":
        raise ValueError('OUTLIER_FILTER_METHOD must be None or "mad".')
    if OUTLIER_TAIL not in {"both", "lower", "upper"}:
        raise ValueError('OUTLIER_TAIL must be "both", "lower", or "upper".')

    valid = np.isfinite(values)
    if weights is not None:
        weights = np.asarray(weights, dtype=float)
        valid &= np.isfinite(weights) & (weights > 0)
    if valid.sum() < OUTLIER_MIN_GROUP_SIZE:
        return mask

    valid_values = values[valid]
    valid_weights = None if weights is None else weights[valid]
    median = weighted_quantile(valid_values, 0.5, valid_weights)
    abs_deviation = np.abs(valid_values - median)
    mad = weighted_quantile(abs_deviation, 0.5, valid_weights)
    robust_scale = 1.4826 * mad
    if not np.isfinite(robust_scale) or robust_scale <= 0:
        return mask

    lower = median - OUTLIER_MAD_Z * robust_scale
    upper = median + OUTLIER_MAD_Z * robust_scale
    valid_mask = np.zeros(len(valid_values), dtype=bool)
    if OUTLIER_TAIL in {"both", "lower"}:
        valid_mask |= valid_values < lower
    if OUTLIER_TAIL in {"both", "upper"}:
        valid_mask |= valid_values > upper
    mask[np.flatnonzero(valid)] = valid_mask
    return mask


def flag_groupwise_outliers(df, group_columns, value_column, weight_column=None):
    # explode() can create duplicate index labels, so assign flags by row position.
    flagged = df.copy().reset_index(drop=True)
    outlier_flags = np.zeros(len(flagged), dtype=bool)
    grouped_positions = flagged.groupby(
        group_columns,
        dropna=False,
        sort=False,
    ).indices
    for positions in grouped_positions.values():
        positions = np.asarray(positions, dtype=int)
        group = flagged.iloc[positions]
        weights = None if weight_column is None else group[weight_column].to_numpy(float)
        outlier_flags[positions] = robust_outlier_mask(
            group[value_column].to_numpy(float),
            weights,
        )
    flagged["is_outlier"] = outlier_flags
    return flagged


## 1. Load And Filter Data


In [20]:
from src.data.filters import (
    filter_graphs_by_blacklist,
    filter_graphs_by_marker_diversity,
    filter_graphs_by_metadata,
    filter_graphs_by_numeric_metadata,
    filter_graphs_by_sphericity,
    load_graph_blacklist_from_dir,
)
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import (
    add_log_metadata_features,
    attach_metadata_to_graphs,
    fill_missing_metadata_for_group,
    infer_global_dim,
    load_aux_metadata_for_dir,
    load_marker_names_from_dir,
    promote_metadata_to_graph_tensors,
    snapshot_graph_metadata,
    strip_graph_metadata,
)
from src.data.preprocessing import interpolate_target_outliers_from_neighbors

data_dir = DATA_ROOT / DATASET_NAME
graphs = load_graph_dataset_from_dir(str(data_dir))
meta = load_aux_metadata_for_dir(str(data_dir))
attach_metadata_to_graphs(graphs, meta, exclude_keys=None)
graphs = select_graph_targets(graphs, target_indices=TARGET_INDICES, inplace=False)
marker_names = load_marker_names_from_dir(str(data_dir))
if marker_names is None:
    marker_names = [f"marker_{i}" for i in range(int(graphs[0].x.size(1)))]
print(f"Loaded {len(graphs)} organoids and {len(marker_names)} markers.")


Loaded 1423 graphs; skipped 0.
Loaded 1423 organoids and 7 markers.


In [21]:
if TIMEPOINT_FILTER_MODE == "rest":
    graphs = filter_graphs_by_metadata(
        graphs, key="timepoint", drop_values={DAY3P5_TIMEPOINT},
        missing="keep", inplace=False, print_summary=True,
    )
elif TIMEPOINT_FILTER_MODE == "day3p5":
    graphs = filter_graphs_by_metadata(
        graphs, key="timepoint", keep_values={DAY3P5_TIMEPOINT},
        missing="drop", inplace=False, print_summary=True,
    )
elif TIMEPOINT_FILTER_MODE not in (None, "all"):
    raise ValueError('TIMEPOINT_FILTER_MODE must be "rest", "day3p5", or "all".')

if FILTER_BLACKLISTED_ORGANOIDS:
    blacklist = load_graph_blacklist_from_dir(data_dir)
    graphs = filter_graphs_by_blacklist(graphs, blacklist, print_summary=True)

if FILL_MISSING_COMPLEXITY:
    graphs = fill_missing_metadata_for_group(
        graphs,
        field="complexity",
        fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
        dataset=MISSING_COMPLEXITY_GROUP["dataset"],
        timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
    )

graphs, spherical = filter_graphs_by_sphericity(
    graphs, max_sphericity=SPHERICITY_MAX, print_summary=True, return_rejected=True,
)
spherical = filter_graphs_by_marker_diversity(
    spherical, min_score=SPHERICAL_MARKER_DIVERSITY_MIN, print_summary=True,
)
graphs = filter_graphs_by_numeric_metadata(
    graphs, key="complexity", min_value=COMPLEXITY_MIN,
    allow_missing=False, inplace=False, print_summary=True,
)
if INTERPOLATE_TARGET_OUTLIERS:
    graphs, outlier_info = interpolate_target_outliers_from_neighbors(
        graphs, clip_quantiles=OUTLIER_CLIP_QUANTILES,
    )
print(f"After filtering: {len(graphs)} organoids.")


filter_graphs_by_metadata(key='timepoint'): kept 1134 / 1423 graphs

dataset          timepoint          kept  total     frac
----------------------------------------------------------
20250929         day3p5                0    289    0.000
20250929         day4                342    342    1.000
20250929         day4p5              104    104    1.000
20250929         day4p5-more         411    411    1.000
20251201         day4p5              277    277    1.000
filter_graphs_by_blacklist(n_keys=6): kept 1128 / 1134 graphs

dataset          timepoint          kept  total     frac
----------------------------------------------------------
20250929         day4                341    342    0.997
20250929         day4p5              104    104    1.000
20250929         day4p5-more         409    411    0.995
20251201         day4p5              274    277    0.989
filter_graphs_by_sphericity(max_sphericity=0.92): kept 813 / 1128 graphs

dataset          timepoint          kept  total  

## 2. Train One Model


In [22]:
from src.data.splits import graph_metadata_key, train_val_split_graphs
from src.data.target_transforms import (
    AsinhStandardizeTransform,
    ChainedTargetTransform,
    GlobalBaselineResidualTransform,
    standardize_graph_global_features,
)
from src.models.gnn import GINCurvature
from src.training.loop import TrainConfig, train
from src.training.losses import WeightedLossTerm, edge_loss_term

g_train, g_val, split_info = train_val_split_graphs(
    graphs,
    val_frac=VAL_FRAC,
    seed=SPLIT_SEED,
    key_fn=graph_metadata_key,
    inplace=False,
)
g_train = copy.deepcopy(g_train)
g_val = copy.deepcopy(g_val)

field_specs = [{
    "meta_keys": ["log_surface_area", "log_volume", "log_volume_over_area", "log_num_cells"],
    "attr_name": "global_feat",
    "kind": "graph_vector",
    "dtype": torch.float32,
}]
if USE_GLOBAL_FEATURES:
    g_train = promote_metadata_to_graph_tensors(
        add_log_metadata_features(g_train, inplace=False), field_specs, inplace=False,
    )
    g_val = promote_metadata_to_graph_tensors(
        add_log_metadata_features(g_val, inplace=False), field_specs, inplace=False,
    )

train_meta_lookup = snapshot_graph_metadata(g_train)
val_meta_lookup = snapshot_graph_metadata(g_val)
g_train = strip_graph_metadata(g_train, inplace=False)
g_val = strip_graph_metadata(g_val, inplace=False)

if USE_GLOBAL_FEATURES:
    standardize_graph_global_features(g_train, g_val, attr_name="global_feat", robust=False)

if SUBTRACT_CONSTANT_GLOBAL_BASELINE:
    target_transform = ChainedTargetTransform([
        GlobalBaselineResidualTransform(num_workers=NUM_WORKERS),
        AsinhStandardizeTransform(robust=True),
    ])
else:
    target_transform = AsinhStandardizeTransform(robust=True)
target_transform.fit(g_train)
target_transform.transform_graphs(g_train, in_place=True)
target_transform.transform_graphs(g_val, in_place=True)

model = GINCurvature(
    n_markers=int(g_train[0].x.size(1)),
    global_dim=infer_global_dim(g_train),
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    residual=RESIDUAL,
    norm=NORM,
)
train_config = TrainConfig(
    lr=LR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    aux_losses=[WeightedLossTerm(
        name="edge", fn=edge_loss_term, weight=EDGE_LOSS_WEIGHT, params=EDGE_LOSS_PARAMS,
    )],
)
model, metrics, history = train(model, g_train, g_val, train_config)
print(metrics)


epoch 001 | train loss -0.0912 mae 0.1127 | val loss -0.1125 mae 0.0995
epoch 002 | train loss -0.1210 mae 0.1009 | val loss -0.1422 mae 0.0873
epoch 003 | train loss -0.1509 mae 0.0908 | val loss -0.1727 mae 0.0762
epoch 004 | train loss -0.1809 mae 0.0828 | val loss -0.2046 mae 0.0667
epoch 005 | train loss -0.2137 mae 0.0771 | val loss -0.2380 mae 0.0600
epoch 006 | train loss -0.2472 mae 0.0737 | val loss -0.2737 mae 0.0547
epoch 007 | train loss -0.2830 mae 0.0717 | val loss -0.3120 mae 0.0514
epoch 008 | train loss -0.3206 mae 0.0711 | val loss -0.3533 mae 0.0491
epoch 009 | train loss -0.3622 mae 0.0712 | val loss -0.3980 mae 0.0482
epoch 010 | train loss -0.4055 mae 0.0719 | val loss -0.4466 mae 0.0479
epoch 011 | train loss -0.4538 mae 0.0728 | val loss -0.4990 mae 0.0473
epoch 012 | train loss -0.5018 mae 0.0732 | val loss -0.5558 mae 0.0459
epoch 013 | train loss -0.5539 mae 0.0736 | val loss -0.6165 mae 0.0453
epoch 014 | train loss -0.6065 mae 0.0744 | val loss -0.6797 mae

In [23]:
from src.inference.predict import predict_targets

y_true, y_pred, _ = predict_targets(
    g_val, model, batch_size=BATCH_SIZE, target_transform=target_transform,
)
y_true = select_target_column(y_true, TARGET_INDEX_FOR_ANALYSIS)
y_pred = select_target_column(y_pred, TARGET_INDEX_FOR_ANALYSIS)
print("Validation MSE:", np.mean((y_pred - y_true) ** 2))
print("Validation MAE:", np.mean(np.abs(y_pred - y_true)))


Validation MSE: 0.0021151356962024886
Validation MAE: 0.031298160149975544


## 3. Build Validation Ego-Subgraphs


In [24]:
from src.data.subgraphs import build_ego_subgraphs_for_dataset

val_subgraphs = build_ego_subgraphs_for_dataset(
    g_val,
    num_hops=NUM_LAYERS,
    max_centers_per_graph=None,
    seed=SUBGRAPH_SEED,
    copy_graph_level_attrs=True,
)
print(f"Built {len(val_subgraphs)} validation ego-subgraphs.")


Built 64649 validation ego-subgraphs.


## 4. Coverage Sample: Mean Heatmaps And Full Distributions

The coverage sample is intentionally not population representative. It is used only for conditional center-marker analyses. When enabled, the robust filter is fit separately within each `(hop, center marker, ablated marker)` group; raw cases and outlier flags are retained in the saved table.


In [ ]:
from src.analysis.perturbation import compute_perturbation_influence_maps
from src.data.subgraph_sampling import sample_subgraphs_coverage, print_sampling_summary

coverage_subgraphs, coverage_info = sample_subgraphs_coverage(
    val_subgraphs,
    marker_names=marker_names,
    k_hops=NUM_LAYERS,
    max_subgraphs=HEATMAP_SAMPLE_SIZE,
    min_center_count=HEATMAP_MIN_CENTER_COUNT,
    min_pair_count=HEATMAP_MIN_PAIR_COUNT,
    seed=SUBGRAPH_SEED,
)
print_sampling_summary(coverage_info, marker_names)
coverage_result = compute_perturbation_influence_maps(
    coverage_subgraphs,
    model,
    marker_names,
    k_hops=NUM_LAYERS,
    mode=PERTURBATION_MODE,
    max_subgraphs=None,
    batch_size=PERTURB_BATCH_SIZE,
    target_index=TARGET_INDEX_FOR_ANALYSIS,
    target_transform=target_transform,
    normalize_by="cases",
    return_case_effects=True,
)
coverage_cases = pd.DataFrame(coverage_result["case_effects"])
coverage_pairs = coverage_cases.explode("center_marker_names").rename(
    columns={"center_marker_names": "center_marker"}
)
coverage_pairs = flag_groupwise_outliers(
    coverage_pairs,
    group_columns=["hop", "center_marker", "source_marker"],
    value_column="delta_mu",
)
coverage_pairs_filtered = coverage_pairs.loc[~coverage_pairs["is_outlier"]].copy()
coverage_pairs.to_csv(TABLES_DIR / "coverage_case_effects_with_outlier_flags.csv", index=False)
print(
    f"Coverage outliers flagged: {int(coverage_pairs['is_outlier'].sum())} / "
    f"{len(coverage_pairs)} exploded center-marker cases"
)
display(coverage_pairs.head())


In [ ]:
from src.plotting.influence_maps import plot_influence_center_resolved


def center_resolved_arrays(pair_df, marker_names, hops):
    n_hops = len(hops)
    n_markers = len(marker_names)
    means = np.full((n_hops, n_markers, n_markers), np.nan, dtype=float)
    counts = np.zeros((n_hops, n_markers, n_markers), dtype=int)
    marker_to_idx = {marker: i for i, marker in enumerate(marker_names)}
    hop_to_idx = {hop: i for i, hop in enumerate(hops)}

    grouped = pair_df.groupby(
        ["hop", "center_marker", "source_marker"],
        dropna=False,
    )["delta_mu"]
    for (hop, center_marker, source_marker), values in grouped:
        if center_marker not in marker_to_idx:
            continue
        hi = hop_to_idx[int(hop)]
        ci = marker_to_idx[center_marker]
        si = int(source_marker)
        finite_values = values[np.isfinite(values)].to_numpy(float)
        if len(finite_values):
            means[hi, ci, si] = finite_values.mean()
            counts[hi, ci, si] = len(finite_values)
    return means, counts


coverage_mean_filtered, coverage_counts_filtered = center_resolved_arrays(
    coverage_pairs_filtered,
    marker_names,
    coverage_result["hops"],
)
fig, axes = plot_influence_center_resolved(
    coverage_mean_filtered,
    coverage_result["hops"],
    marker_names,
    "Mean single-cell ablation effect by center and perturbed marker",
    center_zero=True,
    sort_center=False,
    cmap="RdBu_r",
    counts=coverage_counts_filtered,
    min_count=HEATMAP_DISPLAY_MIN_CASES,
    bad_color="white",
)
save_figure(fig, "coverage_mean_delta_mu_heatmaps")
plt.show()


In [ ]:
def plot_pair_distributions(pair_df, center_marker, marker_names, hops):
    fig, axes = plt.subplots(
        1, len(hops),
        figsize=(max(4.0 * len(hops), 8.0), 4.8),
        sharey=False,
        squeeze=False,
    )
    axes = axes[0]
    subset = pair_df[pair_df["center_marker"] == center_marker]

    for ax, hop in zip(axes, hops):
        hop_df = subset[subset["hop"] == hop]
        positions = []
        distributions = []
        labels = []
        for marker_idx, marker in enumerate(marker_names):
            marker_df = hop_df[hop_df["source_marker"] == marker_idx]
            values = marker_df.loc[~marker_df["is_outlier"], "delta_mu"].dropna().to_numpy()
            n_removed = int(marker_df["is_outlier"].sum())
            if len(values):
                positions.append(marker_idx)
                distributions.append(values)
                suffix = f", out={n_removed}" if n_removed else ""
                labels.append(f"{marker}\n(n={len(values)}{suffix})")

        if distributions:
            parts = ax.violinplot(
                distributions,
                positions=positions,
                widths=0.8,
                showmeans=True,
                showmedians=True,
                showextrema=False,
            )
            for body in parts["bodies"]:
                body.set_facecolor("#4C78A8")
                body.set_edgecolor("#1F2937")
                body.set_alpha(0.65)
        ax.axhline(0.0, color="0.25", linestyle="--", linewidth=1)
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=60, ha="right")
        ax.set_title(f"hop {hop}")
        ax.set_xlabel("ablated marker")

    axes[0].set_ylabel("prediction change (delta mu)")
    filter_label = "unfiltered" if OUTLIER_FILTER_METHOD is None else f"MAD-filtered (z={OUTLIER_MAD_Z:g})"
    fig.suptitle(
        f"Ablation-effect distributions | center marker: {center_marker} | {filter_label}"
    )
    fig.tight_layout()
    return fig, axes


centers_to_plot = marker_names if DISTRIBUTION_CENTER_MARKERS is None else DISTRIBUTION_CENTER_MARKERS
for center_marker in centers_to_plot:
    fig, axes = plot_pair_distributions(
        coverage_pairs, center_marker, marker_names, coverage_result["hops"],
    )
    save_figure(fig, f"coverage_distributions_center_{center_marker}")
    plt.show()


## 4b. Center-Resolved Conversion Analysis

The two conversions reuse the coverage-selected subgraphs. Their center-resolved cases inherit the MAD outlier flag from the matching source-marker ablation case, keyed by subgraph, hop, source marker, and center marker. The conversion effects are not used to fit a second center-resolved outlier rule. Violin plots retain the removal counts so this choice can be inspected directly.


In [ ]:
from src.analysis.perturbation import compute_conversion_influence_maps


def marker_index(marker):
    marker_list = list(marker_names)
    if marker in marker_list:
        return marker_list.index(marker)
    lower = {str(name).lower(): i for i, name in enumerate(marker_list)}
    key = str(marker).lower()
    if key not in lower:
        raise ValueError(f"Marker {marker!r} not found in marker_names.")
    return lower[key]


for spec in CONVERSION_SPECS:
    marker_index(spec["source"])
    marker_index(spec["target"])

ablation_flag_columns = [
    "subgraph_index",
    "hop",
    "source_marker",
    "center_marker",
    "is_outlier",
]
ablation_outlier_flags = coverage_pairs[ablation_flag_columns].rename(
    columns={"is_outlier": "ablation_is_outlier"}
)

coverage_conversion_results = {}
coverage_conversion_pairs = {}
for spec in CONVERSION_SPECS:
    result = compute_conversion_influence_maps(
        coverage_subgraphs,
        model,
        marker_names,
        k_hops=NUM_LAYERS,
        conversion_pairs=[(spec["source"], spec["target"])],
        mode="single",
        max_subgraphs=None,
        batch_size=PERTURB_BATCH_SIZE,
        target_index=TARGET_INDEX_FOR_ANALYSIS,
        target_transform=target_transform,
        normalize_by="cases",
        return_case_effects=True,
    )
    pairs = pd.DataFrame(result["case_effects"]).explode(
        "center_marker_names"
    ).rename(columns={"center_marker_names": "center_marker"})
    pairs = pairs.reset_index(drop=True).merge(
        ablation_outlier_flags,
        on=["subgraph_index", "hop", "source_marker", "center_marker"],
        how="left",
        validate="one_to_one",
    )
    if pairs["ablation_is_outlier"].isna().any():
        raise RuntimeError(
            f"Some {spec['label']} cases could not be matched to their ablation case."
        )
    pairs["is_outlier"] = pairs["ablation_is_outlier"].astype(bool)
    pairs["conversion_key"] = spec["key"]
    pairs["conversion_label"] = spec["label"]
    coverage_conversion_results[spec["key"]] = result
    coverage_conversion_pairs[spec["key"]] = pairs
    pairs.to_csv(
        TABLES_DIR / f"coverage_conversion_{spec['key']}_with_ablation_outlier_flags.csv",
        index=False,
    )
    print(
        f"{spec['label']}: inherited {int(pairs['is_outlier'].sum())} ablation "
        f"outlier flags across {len(pairs)} exploded center-marker cases"
    )


In [ ]:
def conversion_center_hop_arrays(pair_df, marker_names, hops):
    n_hops = len(hops)
    means = np.full((len(marker_names), n_hops), np.nan, dtype=float)
    counts = np.zeros((len(marker_names), n_hops), dtype=int)
    marker_to_idx = {marker: i for i, marker in enumerate(marker_names)}
    hop_to_idx = {int(hop): i for i, hop in enumerate(hops)}

    retained = pair_df.loc[~pair_df["is_outlier"]]
    grouped = retained.groupby(["center_marker", "hop"], dropna=False)["delta_mu"]
    for (center_marker, hop), values in grouped:
        if center_marker not in marker_to_idx:
            continue
        ci = marker_to_idx[center_marker]
        hi = hop_to_idx[int(hop)]
        finite_values = values[np.isfinite(values)].to_numpy(float)
        if len(finite_values):
            means[ci, hi] = finite_values.mean()
            counts[ci, hi] = len(finite_values)
    return means, counts


conversion_heatmaps = {}
finite_values = []
for spec in CONVERSION_SPECS:
    means, counts = conversion_center_hop_arrays(
        coverage_conversion_pairs[spec["key"]],
        marker_names,
        coverage_conversion_results[spec["key"]]["hops"],
    )
    display_values = means.copy()
    display_values[counts < HEATMAP_DISPLAY_MIN_CASES] = np.nan
    conversion_heatmaps[spec["key"]] = {
        "values": display_values,
        "counts": counts,
    }
    finite_values.extend(display_values[np.isfinite(display_values)].tolist())

vmax = max(abs(np.asarray(finite_values))) if finite_values else 1.0
vmax = max(float(vmax), np.finfo(float).eps)
cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad("white")
fig, axes = plt.subplots(
    1,
    len(CONVERSION_SPECS),
    figsize=(5.0 * len(CONVERSION_SPECS) + 1.0, 0.42 * len(marker_names) + 2.8),
    sharey=True,
    constrained_layout=True,
)
axes = np.atleast_1d(axes)
for ax, spec in zip(axes, CONVERSION_SPECS):
    values = conversion_heatmaps[spec["key"]]["values"]
    im = ax.imshow(values, aspect="auto", cmap=cmap, vmin=-vmax, vmax=vmax)
    ax.set_xticks(np.arange(NUM_LAYERS))
    ax.set_xticklabels([f"hop {hop}" for hop in range(1, NUM_LAYERS + 1)])
    ax.set_yticks(np.arange(len(marker_names)))
    ax.set_yticklabels(marker_names)
    ax.set_xlabel("converted source-marker distance")
    ax.set_title(spec["label"])
axes[0].set_ylabel("center marker")
fig.colorbar(im, ax=axes.tolist(), label="mean prediction change (delta mu)", shrink=0.9)
fig.suptitle("Center-resolved conversion effects | inherited ablation MAD filter")
save_figure(fig, "coverage_conversion_center_resolved_combined")
plt.show()


In [ ]:
def plot_conversion_distributions(pair_df, spec, marker_names, hops):
    fig, axes = plt.subplots(
        1,
        len(hops),
        figsize=(max(4.0 * len(hops), 9.0), 4.9),
        sharey=False,
        squeeze=False,
    )
    axes = axes[0]
    for ax, hop in zip(axes, hops):
        hop_df = pair_df[pair_df["hop"] == hop]
        positions = []
        distributions = []
        labels = []
        for center_idx, center_marker in enumerate(marker_names):
            center_df = hop_df[hop_df["center_marker"] == center_marker]
            values = center_df.loc[
                ~center_df["is_outlier"], "delta_mu"
            ].dropna().to_numpy(float)
            removed_values = center_df.loc[
                center_df["is_outlier"], "delta_mu"
            ].dropna().to_numpy(float)
            n_removed = len(removed_values)
            if len(values):
                positions.append(center_idx)
                distributions.append(values)
                suffix = f", out={n_removed}" if n_removed else ""
                labels.append(f"{center_marker}\n(n={len(values)}{suffix})")
                if n_removed:
                    ax.scatter(
                        np.full(n_removed, center_idx),
                        removed_values,
                        marker="x",
                        s=22,
                        linewidths=1.2,
                        color="#D62728",
                        alpha=0.8,
                        zorder=4,
                    )

        if distributions:
            parts = ax.violinplot(
                distributions,
                positions=positions,
                widths=0.8,
                showmeans=True,
                showmedians=True,
                showextrema=False,
            )
            for body in parts["bodies"]:
                body.set_facecolor("#4C78A8")
                body.set_edgecolor("#1F2937")
                body.set_alpha(0.65)
        ax.axhline(0.0, color="0.25", linestyle="--", linewidth=1)
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=60, ha="right")
        ax.set_title(f"hop {hop}")
        ax.set_xlabel("center marker")
    axes[0].set_ylabel("conversion prediction change (delta mu)")
    fig.suptitle(f"{spec['label']} | red x = excluded by source-ablation MAD filter")
    fig.tight_layout()
    return fig, axes


for spec in CONVERSION_SPECS:
    result = coverage_conversion_results[spec["key"]]
    fig, axes = plot_conversion_distributions(
        coverage_conversion_pairs[spec["key"]],
        spec,
        marker_names,
        result["hops"],
    )
    save_figure(fig, f"coverage_conversion_distributions_{spec['key']}")
    plt.show()


## 5. Population Samples

For each weighting scheme:

- `prevalence` estimates \(P(X\text{ present at }d)\) from the representative draw only.
- `conditional_effect` estimates \(E[\Delta\mu\mid X\text{ present at }d]\) from a separate marker-specific sample.
- `population_effect` is their product.

For every marker and hop, the marker-specific sample starts with eligible centers from the representative draw and is topped up from the full validation pool until `MIN_MARKER_COUNT_PER_HOP` is reached, where possible. These top-ups are used only for that marker's conditional effect and never enter prevalence estimates or another marker's sample.


In [ ]:
from src.data.subgraph_sampling import sample_subgraphs_population


def run_population_analysis(weighting):
    population_subgraphs, sample_info = sample_subgraphs_population(
        val_subgraphs,
        max_subgraphs=POPULATION_SAMPLE_SIZE,
        weighting=weighting,
        seed=SUBGRAPH_SEED,
        marker_names=marker_names,
        k_hops=NUM_LAYERS,
        min_marker_count_per_hop=MIN_MARKER_COUNT_PER_HOP,
    )
    print(
        f"{weighting}: population={sample_info['n_population']}, "
        f"groups={len(sample_info['population_group_counts'])}"
    )

    marker_runs = {}
    for marker in marker_names:
        source_by_hop = sample_info["marker_sample_source_indices_by_hop"][marker]
        union_source = []
        seen = set()
        for hop in range(1, NUM_LAYERS + 1):
            for source_idx in source_by_hop[hop]:
                source_idx = int(source_idx)
                if source_idx not in seen:
                    union_source.append(source_idx)
                    seen.add(source_idx)

        source_to_local = {source_idx: i for i, source_idx in enumerate(union_source)}
        local_indices_by_hop = {
            hop: np.asarray(
                [source_to_local[int(source_idx)] for source_idx in source_by_hop[hop]],
                dtype=int,
            )
            for hop in range(1, NUM_LAYERS + 1)
        }
        marker_subgraphs = [val_subgraphs[source_idx] for source_idx in union_source]
        result = compute_perturbation_influence_maps(
            marker_subgraphs,
            model,
            marker_names,
            k_hops=NUM_LAYERS,
            mode="single",
            max_subgraphs=None,
            batch_size=PERTURB_BATCH_SIZE,
            target_index=TARGET_INDEX_FOR_ANALYSIS,
            target_transform=target_transform,
            normalize_by="cases",
            return_case_effects=True,
            source_markers=[marker],
        )
        marker_runs[marker] = {
            "subgraphs": marker_subgraphs,
            "result": result,
            "source_indices": np.asarray(union_source, dtype=int),
            "local_indices_by_hop": local_indices_by_hop,
            "weights_by_hop": sample_info["marker_sample_weights_by_hop"][marker],
        }

    sampled_counts = {
        marker: {
            hop: len(sample_info["marker_sample_source_indices_by_hop"][marker][hop])
            for hop in range(1, NUM_LAYERS + 1)
        }
        for marker in marker_names
    }
    print("Marker-specific conditional sample sizes:", sampled_counts)
    return {
        "population_subgraphs": population_subgraphs,
        "sample_info": sample_info,
        "marker_runs": marker_runs,
    }


population_runs = {
    weighting: run_population_analysis(weighting)
    for weighting in ("cell", "organoid")
}


In [ ]:
from src.graph.neighborhood import compute_hop_rings


def population_marker_prevalence(population_subgraphs, population_weights, marker_idx, hop):
    present = np.zeros(len(population_subgraphs), dtype=bool)
    for i, subgraph in enumerate(population_subgraphs):
        rings = compute_hop_rings(
            subgraph.edge_index,
            int(subgraph.center_idx),
            NUM_LAYERS,
        )
        nodes = rings[hop]
        if len(nodes):
            present[i] = bool((subgraph.x[nodes, marker_idx] > 0.5).any().item())
    return float(np.asarray(population_weights, dtype=float)[present].sum())


def population_effect_table(run, marker_names):
    population_subgraphs = run["population_subgraphs"]
    sample_info = run["sample_info"]
    marker_runs = run["marker_runs"]
    population_weights = np.asarray(sample_info["population_weights"], dtype=float)

    rows = []
    for source_idx, marker in enumerate(marker_names):
        marker_run = marker_runs[marker]
        cases = pd.DataFrame(marker_run["result"]["case_effects"])
        for hop in range(1, NUM_LAYERS + 1):
            sample_positions = marker_run["local_indices_by_hop"][hop]
            sample_weights = np.asarray(marker_run["weights_by_hop"][hop], dtype=float)
            hop_cases = cases[
                (cases["hop"] == hop)
                & (cases["source_marker"] == source_idx)
                & (cases["subgraph_index"].isin(sample_positions))
            ].copy()
            weight_map = dict(zip(sample_positions.tolist(), sample_weights.tolist()))
            hop_cases["sample_weight"] = hop_cases["subgraph_index"].map(weight_map).astype(float)
            hop_cases["is_outlier"] = robust_outlier_mask(
                hop_cases["delta_mu"].to_numpy(float),
                hop_cases["sample_weight"].to_numpy(float),
            )
            kept_cases = hop_cases.loc[~hop_cases["is_outlier"]]
            conditional = weighted_mean(
                kept_cases["delta_mu"].to_numpy(float),
                kept_cases["sample_weight"].to_numpy(float),
            )
            prevalence = population_marker_prevalence(
                population_subgraphs,
                population_weights,
                source_idx,
                hop,
            )
            n_topup = len(
                sample_info["marker_topup_source_indices_by_hop"][marker][hop]
            )
            rows.append({
                "weighting": sample_info["weighting"],
                "hop": hop,
                "source_marker": source_idx,
                "marker": marker,
                "prevalence": prevalence,
                "conditional_effect": conditional,
                "population_effect": prevalence * conditional,
                "conditional_cases": len(kept_cases),
                "outlier_cases": int(hop_cases["is_outlier"].sum()),
                "unfiltered_conditional_cases": len(hop_cases),
                "topup_cases": n_topup,
                "available_cases": sample_info["marker_available_by_hop"][marker][hop],
            })
    return pd.DataFrame(rows)


population_effects = pd.concat(
    [population_effect_table(run, marker_names) for run in population_runs.values()],
    ignore_index=True,
)
population_effects.to_csv(TABLES_DIR / "population_ablation_effects.csv", index=False)
display(population_effects.head(12))


In [ ]:
def plot_population_curves(effect_df, marker_names):
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
    columns = [
        ("conditional_effect", r"$E[\Delta\mu \mid X\ present\ at\ d]$"),
        ("population_effect", r"$P(X\ present\ at\ d) E[\Delta\mu \mid X\ present\ at\ d]$"),
    ]
    for row, weighting in enumerate(("cell", "organoid")):
        subset = effect_df[effect_df["weighting"] == weighting]
        for col, (value_col, title) in enumerate(columns):
            ax = axes[row, col]
            for marker in marker_names:
                marker_df = subset[subset["marker"] == marker].sort_values("hop")
                ax.plot(
                    marker_df["hop"], marker_df[value_col],
                    marker="o", linewidth=1.6, label=marker,
                )
            ax.axhline(0.0, color="0.25", linestyle="--", linewidth=1)
            ax.set_title(f"Average {weighting} | {title}")
            ax.set_xlabel("distance from center (hop)")
            ax.set_ylabel("prediction change")
            ax.set_xticks(sorted(subset["hop"].unique()))
    axes[0, 1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    return fig, axes

fig, axes = plot_population_curves(population_effects, marker_names)
save_figure(fig, "population_conditional_and_prevalence_weighted_effects")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, weighting in zip(axes, ("cell", "organoid")):
    subset = population_effects[population_effects["weighting"] == weighting]
    for marker in marker_names:
        marker_df = subset[subset["marker"] == marker].sort_values("hop")
        ax.plot(marker_df["hop"], marker_df["prevalence"], marker="o", label=marker)
    ax.set_title(f"Marker prevalence | average {weighting}")
    ax.set_xlabel("distance from center (hop)")
    ax.set_ylabel(r"$P(X\ present\ at\ d)$")
    ax.set_ylim(bottom=0)
    ax.set_xticks(sorted(subset["hop"].unique()))
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
save_figure(fig, "population_marker_prevalence")
plt.show()


## 5b. Population Conversion Effects

Each conversion reuses the source marker's existing topped-up conditional sample. Source-marker prevalence still comes only from the representative population draw. Unlike the center-resolved diagnostic above, population conversion effects use their own weighted MAD filter within each `(weighting, conversion, hop)` group.


In [ ]:
def run_population_conversions(population_runs, conversion_specs):
    conversion_runs = {}
    for weighting, run in population_runs.items():
        conversion_runs[weighting] = {}
        for spec in conversion_specs:
            marker_run = run["marker_runs"][spec["source"]]
            result = compute_conversion_influence_maps(
                marker_run["subgraphs"],
                model,
                marker_names,
                k_hops=NUM_LAYERS,
                conversion_pairs=[(spec["source"], spec["target"])],
                mode="single",
                max_subgraphs=None,
                batch_size=PERTURB_BATCH_SIZE,
                target_index=TARGET_INDEX_FOR_ANALYSIS,
                target_transform=target_transform,
                normalize_by="cases",
                return_case_effects=True,
            )
            conversion_runs[weighting][spec["key"]] = result
    return conversion_runs


population_conversion_runs = run_population_conversions(
    population_runs,
    CONVERSION_SPECS,
)


In [ ]:
def population_conversion_effect_table(population_runs, conversion_runs, specs):
    rows = []
    for weighting, run in population_runs.items():
        sample_info = run["sample_info"]
        population_subgraphs = run["population_subgraphs"]
        population_weights = np.asarray(sample_info["population_weights"], dtype=float)
        for spec in specs:
            source_idx = marker_index(spec["source"])
            marker_run = run["marker_runs"][spec["source"]]
            cases = pd.DataFrame(conversion_runs[weighting][spec["key"]]["case_effects"])
            for hop in range(1, NUM_LAYERS + 1):
                sample_positions = marker_run["local_indices_by_hop"][hop]
                sample_weights = np.asarray(marker_run["weights_by_hop"][hop], dtype=float)
                hop_cases = cases[
                    (cases["hop"] == hop)
                    & (cases["source_marker"] == source_idx)
                    & (cases["subgraph_index"].isin(sample_positions))
                ].copy()
                weight_map = dict(zip(sample_positions.tolist(), sample_weights.tolist()))
                hop_cases["sample_weight"] = hop_cases["subgraph_index"].map(
                    weight_map
                ).astype(float)
                hop_cases["is_outlier"] = robust_outlier_mask(
                    hop_cases["delta_mu"].to_numpy(float),
                    hop_cases["sample_weight"].to_numpy(float),
                )
                retained = hop_cases.loc[~hop_cases["is_outlier"]]
                conditional = weighted_mean(
                    retained["delta_mu"].to_numpy(float),
                    retained["sample_weight"].to_numpy(float),
                )
                prevalence = population_marker_prevalence(
                    population_subgraphs,
                    population_weights,
                    source_idx,
                    hop,
                )
                rows.append({
                    "weighting": weighting,
                    "conversion_key": spec["key"],
                    "conversion": spec["label"],
                    "source_marker": spec["source"],
                    "target_marker": spec["target"],
                    "hop": hop,
                    "prevalence": prevalence,
                    "conditional_effect": conditional,
                    "population_effect": prevalence * conditional,
                    "conditional_cases": len(retained),
                    "outlier_cases": int(hop_cases["is_outlier"].sum()),
                    "unfiltered_conditional_cases": len(hop_cases),
                })
    return pd.DataFrame(rows)


population_conversion_effects = population_conversion_effect_table(
    population_runs,
    population_conversion_runs,
    CONVERSION_SPECS,
)
population_conversion_effects.to_csv(
    TABLES_DIR / "population_conversion_effects.csv",
    index=False,
)
display(population_conversion_effects)


In [ ]:
def plot_population_conversion_curves(conversion_df, ablation_df, specs):
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5), sharex=True)
    columns = [
        ("conditional_effect", r"$E[\Delta\mu \mid X\ present\ at\ d]$"),
        ("population_effect", r"$P(X\ present\ at\ d)E[\Delta\mu \mid X\ present\ at\ d]$"),
    ]
    colors = ["#4C78A8", "#E45756"]
    for row, weighting in enumerate(("cell", "organoid")):
        weighting_conversion = conversion_df[conversion_df["weighting"] == weighting]
        weighting_ablation = ablation_df[ablation_df["weighting"] == weighting]
        for col, (value_column, title) in enumerate(columns):
            ax = axes[row, col]
            for color, spec in zip(colors, specs):
                conversion_curve = weighting_conversion[
                    weighting_conversion["conversion_key"] == spec["key"]
                ].sort_values("hop")
                ax.plot(
                    conversion_curve["hop"],
                    conversion_curve[value_column],
                    marker="o",
                    linewidth=2.0,
                    color=color,
                    label=spec["label"],
                )
                ablation_curve = weighting_ablation[
                    weighting_ablation["marker"] == spec["source"]
                ].sort_values("hop")
                ax.plot(
                    ablation_curve["hop"],
                    ablation_curve[value_column],
                    marker="s",
                    linewidth=1.8,
                    linestyle="--",
                    color=color,
                    alpha=0.85,
                    label=f"{spec['source']} -> 0",
                )
            ax.axhline(0.0, color="0.25", linestyle="--", linewidth=1)
            ax.set_title(f"Average {weighting} | {title}")
            ax.set_xlabel("distance from center (hop)")
            ax.set_ylabel("prediction change")
            ax.set_xticks(range(1, NUM_LAYERS + 1))
            ax.grid(axis="y", alpha=0.2)
    axes[0, 1].legend(frameon=False, loc="best", ncol=2)
    fig.tight_layout()
    return fig, axes


fig, axes = plot_population_conversion_curves(
    population_conversion_effects,
    population_effects,
    CONVERSION_SPECS,
)
save_figure(fig, "population_conversion_conditional_and_weighted_effects")
plt.show()


## 6. Sampling Diagnostics


In [ ]:
diagnostics = []
for weighting, run in population_runs.items():
    sample_info = run["sample_info"]
    for marker in marker_names:
        for hop in range(1, NUM_LAYERS + 1):
            diagnostics.append({
                "weighting": weighting,
                "marker": marker,
                "hop": hop,
                "n_population": sample_info["n_population"],
                "available": sample_info["marker_available_by_hop"][marker][hop],
                "sampled": len(
                    sample_info["marker_sample_source_indices_by_hop"][marker][hop]
                ),
                "topped_up": len(
                    sample_info["marker_topup_source_indices_by_hop"][marker][hop]
                ),
            })
diagnostics_df = pd.DataFrame(diagnostics)
diagnostics_df.to_csv(TABLES_DIR / "sampling_diagnostics.csv", index=False)
display(diagnostics_df)
print("Saved analysis to", SAVE_DIR)


## 7. Physical Cell Reuse Across Sampled Subgraphs

A physical cell is identified by `(graph_idx, original node index)`. For each sampling design, its reuse count is the number of distinct sampled ego-subgraphs whose `orig_nodes` contains that cell. Marker-positive cells that never appear are retained in the diagnostic table and reported as `zero` in each panel; the histogram itself shows represented cells so the reuse tail remains visible.


In [ ]:
from collections import Counter


def subgraph_membership_counts(sampled_subgraphs):
    counts = Counter()
    for subgraph in sampled_subgraphs:
        if not hasattr(subgraph, "graph_idx") or not hasattr(subgraph, "orig_nodes"):
            raise ValueError("Expected sampled subgraphs with graph_idx and orig_nodes.")
        graph_idx = int(subgraph.graph_idx)
        original_nodes = subgraph.orig_nodes.detach().cpu().numpy().astype(
            int,
            copy=False,
        ).reshape(-1)
        for node_idx in original_nodes:
            counts[(graph_idx, int(node_idx))] += 1
    return counts


def marker_cell_reuse_table(graphs, sampled_subgraphs, marker_names, *, sample_name):
    membership = subgraph_membership_counts(sampled_subgraphs)
    rows = []
    for graph_idx, graph in enumerate(graphs):
        marker_positive = np.asarray(graph.x.detach().cpu()) > 0.5
        organoid = getattr(graph, "organoid_str", f"graph_{graph_idx}")
        for marker_idx, marker in enumerate(marker_names):
            for node_idx in np.flatnonzero(marker_positive[:, marker_idx]):
                rows.append({
                    "sample": sample_name,
                    "graph_idx": graph_idx,
                    "organoid_str": organoid,
                    "node_idx": int(node_idx),
                    "marker": marker,
                    "reuse_count": int(membership.get((graph_idx, int(node_idx)), 0)),
                })
    return pd.DataFrame(rows)


def marker_specific_reuse_table(graphs, marker_runs, marker_names, *, sample_name):
    tables = []
    for marker in marker_names:
        table = marker_cell_reuse_table(
            graphs,
            marker_runs[marker]["subgraphs"],
            marker_names,
            sample_name=sample_name,
        )
        tables.append(table[table["marker"] == marker])
    return pd.concat(tables, ignore_index=True)


def plot_marker_reuse_histograms(reuse_df, marker_names, title, filename):
    n_markers = len(marker_names)
    n_cols = min(3, n_markers)
    n_rows = int(np.ceil(n_markers / n_cols))
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.4 * n_cols, 3.5 * n_rows),
        squeeze=False,
    )
    axes = axes.ravel()

    for ax, marker in zip(axes, marker_names):
        marker_counts = reuse_df.loc[
            reuse_df["marker"] == marker,
            "reuse_count",
        ].to_numpy(int)
        represented = marker_counts[marker_counts > 0]
        n_zero = int((marker_counts == 0).sum())

        if len(represented):
            max_count = int(represented.max())
            bins = np.arange(0.5, max_count + 1.5, 1.0)
            ax.hist(
                represented,
                bins=bins,
                color="#4C78A8",
                edgecolor="white",
                linewidth=0.5,
            )
            median = float(np.median(represented))
            p95 = float(np.quantile(represented, 0.95))
            ax.axvline(p95, color="#E45756", linestyle="--", linewidth=1.5)
            stats = (
                f"cells={len(marker_counts):,} | represented={len(represented):,} | zero={n_zero:,}\n"
                f"median={median:.1f} | p95={p95:.1f} | max={max_count}"
            )
        else:
            stats = f"cells={len(marker_counts):,} | represented=0 | zero={n_zero:,}"
            ax.text(0.5, 0.5, "No represented cells", transform=ax.transAxes, ha="center")

        ax.set_title(marker)
        ax.set_xlabel("sampled subgraphs containing cell")
        ax.set_ylabel("marker-positive cells")
        ax.text(
            0.02,
            0.98,
            stats,
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=8.5,
            bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "0.8"},
        )

    for ax in axes[n_markers:]:
        ax.set_visible(False)
    fig.suptitle(title)
    fig.tight_layout()
    save_figure(fig, filename)
    plt.show()
    return fig, axes


In [ ]:
coverage_reuse_df = marker_cell_reuse_table(
    g_val,
    coverage_subgraphs,
    marker_names,
    sample_name="coverage",
)
coverage_reuse_df.to_csv(TABLES_DIR / "coverage_physical_cell_reuse.csv", index=False)
plot_marker_reuse_histograms(
    coverage_reuse_df,
    marker_names,
    "Physical-cell reuse in the center-resolved coverage sample",
    "coverage_physical_cell_reuse_histograms",
)


In [ ]:
population_reuse_tables = []
for weighting, run in population_runs.items():
    reuse_df = marker_specific_reuse_table(
        g_val,
        run["marker_runs"],
        marker_names,
        sample_name=f"population_{weighting}_marker_specific",
    )
    population_reuse_tables.append(reuse_df)
    reuse_df.to_csv(
        TABLES_DIR / f"population_{weighting}_marker_specific_physical_cell_reuse.csv",
        index=False,
    )
    plot_marker_reuse_histograms(
        reuse_df,
        marker_names,
        f"Physical-cell reuse in marker-specific samples | average {weighting}",
        f"population_{weighting}_marker_specific_physical_cell_reuse_histograms",
    )

population_reuse_df = pd.concat(population_reuse_tables, ignore_index=True)


## 8. Hop-Resolved Physical Cell Reuse

These histograms resolve reuse by exact graph distance from the sampled center. Rows are marker types and columns are hop distances. For marker-specific population samples, each column uses only the subgraphs assigned to that marker at that particular hop, including its hop-specific top-ups.


In [ ]:
def hop_membership_counts(sampled_subgraphs, hops, *, subgraph_indices_by_hop=None):
    counts_by_hop = {int(hop): Counter() for hop in hops}
    for hop in hops:
        hop = int(hop)
        if subgraph_indices_by_hop is None:
            selected_indices = range(len(sampled_subgraphs))
        else:
            selected_indices = subgraph_indices_by_hop[hop]

        for subgraph_idx in selected_indices:
            subgraph = sampled_subgraphs[int(subgraph_idx)]
            if not hasattr(subgraph, "graph_idx") or not hasattr(subgraph, "orig_nodes"):
                raise ValueError("Expected sampled subgraphs with graph_idx and orig_nodes.")
            rings = compute_hop_rings(
                subgraph.edge_index,
                int(subgraph.center_idx),
                max(hops),
            )
            local_nodes = np.asarray(rings[hop], dtype=int).reshape(-1)
            if len(local_nodes) == 0:
                continue
            original_nodes = subgraph.orig_nodes.detach().cpu().numpy().astype(
                int,
                copy=False,
            )
            graph_idx = int(subgraph.graph_idx)
            for node_idx in original_nodes[local_nodes]:
                counts_by_hop[hop][(graph_idx, int(node_idx))] += 1
    return counts_by_hop


def marker_hop_reuse_table(
    graphs,
    sampled_subgraphs,
    marker_names,
    hops,
    *,
    sample_name,
    subgraph_indices_by_hop=None,
    markers_to_include=None,
):
    counts_by_hop = hop_membership_counts(
        sampled_subgraphs,
        hops,
        subgraph_indices_by_hop=subgraph_indices_by_hop,
    )
    marker_indices = {
        marker: marker_index(marker)
        for marker in (marker_names if markers_to_include is None else markers_to_include)
    }
    rows = []
    for graph_idx, graph in enumerate(graphs):
        marker_positive = graph.x.detach().cpu().numpy() > 0.5
        organoid = getattr(graph, "organoid_str", f"graph_{graph_idx}")
        for marker, marker_idx in marker_indices.items():
            positive_nodes = np.flatnonzero(marker_positive[:, marker_idx])
            for hop in hops:
                membership = counts_by_hop[int(hop)]
                for node_idx in positive_nodes:
                    rows.append({
                        "sample": sample_name,
                        "graph_idx": graph_idx,
                        "organoid_str": organoid,
                        "node_idx": int(node_idx),
                        "marker": marker,
                        "hop": int(hop),
                        "reuse_count": int(
                            membership.get((graph_idx, int(node_idx)), 0)
                        ),
                    })
    return pd.DataFrame(rows)


def marker_specific_hop_reuse_table(graphs, marker_runs, marker_names, hops, *, sample_name):
    tables = []
    for marker in marker_names:
        marker_run = marker_runs[marker]
        table = marker_hop_reuse_table(
            graphs,
            marker_run["subgraphs"],
            marker_names,
            hops,
            sample_name=sample_name,
            subgraph_indices_by_hop=marker_run["local_indices_by_hop"],
            markers_to_include=[marker],
        )
        tables.append(table)
    return pd.concat(tables, ignore_index=True)


In [ ]:
def plot_marker_hop_reuse_grid(reuse_df, marker_names, hops, title, filename):
    fig, axes = plt.subplots(
        len(marker_names),
        len(hops),
        figsize=(3.7 * len(hops), 2.8 * len(marker_names)),
        squeeze=False,
    )

    for row, marker in enumerate(marker_names):
        for col, hop in enumerate(hops):
            ax = axes[row, col]
            counts = reuse_df.loc[
                (reuse_df["marker"] == marker) & (reuse_df["hop"] == hop),
                "reuse_count",
            ].to_numpy(int)
            represented = counts[counts > 0]
            n_zero = int((counts == 0).sum())

            if len(represented):
                max_count = int(represented.max())
                bins = np.arange(0.5, max_count + 1.5, 1.0)
                ax.hist(
                    represented,
                    bins=bins,
                    color="#4C78A8",
                    edgecolor="white",
                    linewidth=0.45,
                )
                median = float(np.median(represented))
                p95 = float(np.quantile(represented, 0.95))
                ax.axvline(p95, color="#E45756", linestyle="--", linewidth=1.3)
                stats = (
                    f"repr={len(represented):,} | zero={n_zero:,}\n"
                    f"med={median:.1f} | p95={p95:.1f} | max={max_count}"
                )
            else:
                stats = f"repr=0 | zero={n_zero:,}"
                ax.text(
                    0.5,
                    0.5,
                    "No represented cells",
                    transform=ax.transAxes,
                    ha="center",
                    va="center",
                    fontsize=8,
                )

            if row == 0:
                ax.set_title(f"hop {hop}")
            if col == 0:
                ax.set_ylabel(f"{marker}\nmarker-positive cells")
            if row == len(marker_names) - 1:
                ax.set_xlabel("subgraphs containing cell at this hop")
            ax.text(
                0.02,
                0.98,
                stats,
                transform=ax.transAxes,
                va="top",
                ha="left",
                fontsize=7.5,
                bbox={"facecolor": "white", "alpha": 0.82, "edgecolor": "0.82"},
            )

    fig.suptitle(title)
    fig.tight_layout()
    save_figure(fig, filename)
    plt.show()
    return fig, axes


In [ ]:
hops = list(range(1, NUM_LAYERS + 1))
coverage_hop_reuse_df = marker_hop_reuse_table(
    g_val,
    coverage_subgraphs,
    marker_names,
    hops,
    sample_name="coverage",
)
coverage_hop_reuse_df.to_csv(
    TABLES_DIR / "coverage_physical_cell_reuse_by_hop.csv",
    index=False,
)
plot_marker_hop_reuse_grid(
    coverage_hop_reuse_df,
    marker_names,
    hops,
    "Physical-cell reuse by exact distance | center-resolved coverage sample",
    "coverage_physical_cell_reuse_by_hop",
)


In [ ]:
population_hop_reuse_tables = []
for weighting, run in population_runs.items():
    reuse_df = marker_specific_hop_reuse_table(
        g_val,
        run["marker_runs"],
        marker_names,
        hops,
        sample_name=f"population_{weighting}_marker_specific",
    )
    population_hop_reuse_tables.append(reuse_df)
    reuse_df.to_csv(
        TABLES_DIR / f"population_{weighting}_marker_specific_physical_cell_reuse_by_hop.csv",
        index=False,
    )
    plot_marker_hop_reuse_grid(
        reuse_df,
        marker_names,
        hops,
        f"Physical-cell reuse by exact distance | average {weighting}",
        f"population_{weighting}_marker_specific_physical_cell_reuse_by_hop",
    )

population_hop_reuse_df = pd.concat(population_hop_reuse_tables, ignore_index=True)
